# 09 — Decision Trees, Random Forests & Boosting
### From Greedy Partitions to Multi-Class Boosting and XGBoost on CUDA

---

> **Position in the series:**
> `01_statistical_learning_theory` → `02_pac_and_vc_dimensions` → `03_linear_algebra_and_geometry` → `04_pca` → `05_gaussian_mixture_models` → `06_linear_regression` → `07_logistic_regression` → `08_svm_and_kernel_methods` → **`09_decision_trees_random_forest_boosting`** → `10_model_selection_and_validation`

---
## Structure

**Part I — Decision Trees**
1. From Linear Models to Recursive Partitions
2. Node Impurity: Misclassification, Entropy, Gini
3. Greedy Tree Growing
4. Regression Trees
5. Overfitting, Instability and Cost-Complexity Pruning

---
## 0. Preview — The Models in One Page

Every model so far (`06`, `07`, `08`) was a **linear** function of a fixed set of features. This notebook builds models out of *partitions*.

**A decision tree** splits the input space with a sequence of questions $x_j\le\theta$ and predicts a constant in each resulting cell. It needs no feature scaling and handles interactions, but a single tree has **high variance**.

**Two ways of combining trees** fix this, and they attack different terms of the bias-variance decomposition of `01`, §8:

| | Trees | Trained | Mainly reduces |
|---|---|---|---|
| **Random forest** | deep (low bias, high variance) | independently, in parallel | variance |
| **Boosting** | shallow (high bias, low variance) | sequentially, each correcting the previous ones | bias (and, in practice, variance too) |

**The central result of Part III** (Mohri et al. [1], §7.2.2): AdaBoost, designed to turn weak learners into strong ones, *coincides exactly* with coordinate descent on the exponential objective

$$F(\bar{\boldsymbol\alpha})=\frac1m\sum_{i=1}^m e^{-y_i\sum_{j=1}^N\bar\alpha_jh_j(x_i)},$$

a convex upper bound on the training 0-1 error. Part IV generalizes the idea: gradient boosting replaces coordinate descent with functional gradient descent on any differentiable loss, and XGBoost adds second-order information and explicit regularization.

**Notation.** Part III follows [1]: $m$ training points, a set of $N$ base classifiers, $T$ rounds, $k$ classes. Elsewhere the data are $(\mathbf{x}_i,y_i)$ with $d$ features.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

BLUE, RED, TEAL, ORANGE, PURPLE = '#457B9D', '#E63946', '#2A9D8F', '#F4A261', '#6A0572'

# ---------------------------------------------------------------- data and evaluation helpers
def make_moons(n, noise, seed):
    # Two interleaving half-moons (as in 08), labels in {0, 1}
    rng = np.random.default_rng(seed)
    t1, t2 = rng.uniform(0, np.pi, n), rng.uniform(0, np.pi, n)
    A = np.column_stack([np.cos(t1), np.sin(t1)])
    B = np.column_stack([1 - np.cos(t2), 0.5 - np.sin(t2)])
    return np.vstack([A, B]) + rng.normal(0, noise, (2*n, 2)), np.concatenate([np.zeros(n, int), np.ones(n, int)])

def train_test_split_np(X, y, test_frac, seed):
    perm = np.random.default_rng(seed).permutation(len(y))
    n_te = int(test_frac * len(y))
    te, tr = perm[:n_te], perm[n_te:]
    return X[tr], X[te], y[tr], y[te]

def stratified_kfold(y, K, seed):
    # K folds with (almost) the same class proportions; returns a list of (train_idx, val_idx)
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(K)]
    for c in np.unique(y):
        for k, part in enumerate(np.array_split(rng.permutation(np.where(y == c)[0]), K)):
            folds[k].append(part)
    folds = [np.concatenate(f) for f in folds]
    return [(np.concatenate(folds[:k] + folds[k+1:]), folds[k]) for k in range(K)]

def plot_regions(ax, predict, X, y, title, lim=(-1.7, 2.7, -1.2, 1.7)):
    xx, yy = np.meshgrid(np.linspace(lim[0], lim[1], 300), np.linspace(lim[2], lim[3], 300))
    Z = predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=[BLUE, RED], alpha=0.18)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.2)
    ax.scatter(*X[y == 0].T, s=14, color=BLUE); ax.scatter(*X[y == 1].T, s=14, color=RED)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

# ---------------------------------------------------------------- Section 2: impurity
def impurity(P, criterion):
    # P: (..., K) class proportions
    if criterion == 'gini':
        return np.sum(P * (1 - P), axis=-1)
    if criterion == 'entropy':
        return -np.sum(P * np.log2(np.where(P > 0, P, 1.0)), axis=-1)   # 0 log 0 = 0
    return 1 - P.max(axis=-1)                                           # misclassification

# ---------------------------------------------------------------- Sections 3-5: decision tree
class DecisionTree:
    # Greedy CART tree. criterion: 'gini', 'entropy', 'misclassification' or 'mse' (regression).
    # max_features: number of features examined at each split (Section 7); None = all of them.
    def __init__(self, criterion='gini', max_depth=None, min_samples_leaf=1, max_features=None, seed=0, classes=None):
        self.criterion, self.max_depth, self.min_samples_leaf = criterion, max_depth, min_samples_leaf
        self.max_features, self.rng, self.classes_ = max_features, np.random.default_rng(seed), classes
        self.regression = criterion == 'mse'

    def fit(self, X, y):
        self.X_ = X
        if self.regression:
            self.T_ = np.asarray(y, float)[:, None]
        else:
            self.classes_ = np.unique(y) if self.classes_ is None else self.classes_
            self.T_ = (np.asarray(y)[:, None] == self.classes_).astype(float)    # one-hot
        self.feature, self.threshold, self.left, self.right = [], [], [], []
        self.value, self.imp, self.n_node = [], [], []
        self._grow(np.arange(len(X)), 0)
        for a in ['feature', 'threshold', 'left', 'right', 'value', 'imp', 'n_node']:
            setattr(self, a, np.array(getattr(self, a)))
        self.collapsed = np.zeros(len(self.left), bool)     # pruned nodes behave like leaves
        del self.X_, self.T_
        return self

    def _node_stats(self, idx):
        T = self.T_[idx]
        if self.regression:
            return T.mean(axis=0), T.var()
        P = T.mean(axis=0)
        return P, impurity(P, self.criterion)

    def _grow(self, idx, depth):
        node = len(self.left)
        val, imp = self._node_stats(idx)
        for lst, v in [(self.feature, -1), (self.threshold, 0.0), (self.left, -1), (self.right, -1),
                       (self.value, val), (self.imp, imp), (self.n_node, len(idx))]:
            lst.append(v)
        if imp <= 1e-12 or (self.max_depth is not None and depth >= self.max_depth):
            return node
        j, thr = self._best_split(idx)
        if j is None:
            return node
        mask = self.X_[idx, j] <= thr
        self.feature[node], self.threshold[node] = j, thr
        self.left[node] = self._grow(idx[mask], depth + 1)
        self.right[node] = self._grow(idx[~mask], depth + 1)
        return node

    def _best_split(self, idx):
        # For each candidate feature: sort, then evaluate every threshold in O(n) with cumulative sums
        n, d = len(idx), self.X_.shape[1]
        msl = self.min_samples_leaf
        pos = np.arange(1, n)                              # number of points sent to the left child
        feats = np.arange(d) if self.max_features is None else self.rng.permutation(d)
        best, best_j, best_thr, visited = np.inf, None, None, 0
        for j in feats:
            if self.max_features is not None and visited >= self.max_features and best_j is not None:
                break
            x = self.X_[idx, j]
            order = np.argsort(x, kind='stable')
            xs = x[order]
            if xs[0] == xs[-1]:
                continue                                    # constant feature in this node
            visited += 1
            valid = (xs[1:] > xs[:-1]) & (pos >= msl) & (n - pos >= msl)
            if not valid.any():
                continue
            Ts = self.T_[idx][order]
            SL = np.cumsum(Ts, axis=0)[:-1]
            S = SL[-1] + Ts[-1]
            nl, nr = pos[:, None], (n - pos)[:, None]
            if self.regression:                             # weighted child variance, up to a constant
                child = -(SL[:, 0]**2 / pos + (S[0] - SL[:, 0])**2 / (n - pos)) / n
            else:
                child = (pos * impurity(SL / nl, self.criterion) + (n - pos) * impurity((S - SL) / nr, self.criterion)) / n
            child = np.where(valid, child, np.inf)
            k = np.argmin(child)
            if child[k] < best - 1e-12:
                best, best_j, best_thr = child[k], j, xs[k] / 2 + xs[k + 1] / 2
        return best_j, best_thr

    def apply(self, X):
        # Index of the leaf reached by each row of X
        node, active = np.zeros(len(X), int), np.arange(len(X))
        while active.size:
            n = node[active]
            internal = (self.left[n] >= 0) & ~self.collapsed[n]
            active, n = active[internal], n[internal]
            go_left = X[active, self.feature[n]] <= self.threshold[n]
            node[active] = np.where(go_left, self.left[n], self.right[n])
        return node

    def predict_proba(self, X):
        return self.value[self.apply(X)]

    def predict(self, X):
        v = self.predict_proba(X)
        return v[:, 0] if self.regression else self.classes_[np.argmax(v, axis=1)]

    def n_leaves(self):
        reach, stack = 0, [0]
        while stack:
            n = stack.pop()
            if self.left[n] < 0 or self.collapsed[n]:
                reach += 1
            else:
                stack += [self.left[n], self.right[n]]
        return reach

def pruning_path(tree):
    # Weakest-link pruning. Returns the increasing alphas and, for each, the mask of collapsed nodes.
    # alpha = lambda / m, where lambda is the parameter of Mohri et al., eq. (9.15).
    m = tree.n_node[0]
    R_node = tree.n_node / m * tree.imp
    collapsed = np.zeros(len(tree.left), bool)
    alphas, masks = [0.0], [collapsed.copy()]
    while not collapsed[0] and tree.left[0] >= 0:
        R_sub, L_sub, r = R_node.copy(), np.ones(len(R_node)), np.full(len(R_node), np.inf)
        for n in range(len(R_node) - 1, -1, -1):            # children have larger ids than their parent
            if tree.left[n] >= 0 and not collapsed[n]:
                a, b = tree.left[n], tree.right[n]
                R_sub[n], L_sub[n] = R_sub[a] + R_sub[b], L_sub[a] + L_sub[b]
                r[n] = (R_node[n] - R_sub[n]) / (L_sub[n] - 1)
        # only nodes still reachable from the root compete
        reachable, stack = np.zeros(len(R_node), bool), [0]
        while stack:
            n = stack.pop(); reachable[n] = True
            if tree.left[n] >= 0 and not collapsed[n]:
                stack += [tree.left[n], tree.right[n]]
        r[~reachable] = np.inf
        alpha = max(r.min(), 0.0)
        collapsed = collapsed | (r <= r.min() + 1e-12)
        alphas.append(alpha); masks.append(collapsed.copy())
    return np.array(alphas), masks

# ---------------------------------------------------------------- Sections 6-8: bagging and forests
class RandomForest:
    # B trees on bootstrap samples, max_features candidates per split. max_features=None gives bagging.
    def __init__(self, B=100, max_features='sqrt', criterion='gini', min_samples_leaf=1, seed=0):
        self.B, self.max_features, self.criterion, self.min_samples_leaf = B, max_features, criterion, min_samples_leaf
        self.seed = seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        mf = {'sqrt': max(1, int(np.sqrt(d))), None: None}.get(self.max_features, self.max_features)
        self.regression = self.criterion == 'mse'
        self.classes_ = None if self.regression else np.unique(y)
        K = 1 if self.regression else len(self.classes_)
        self.trees, self.boots = [], []
        oob_sum, oob_cnt = np.zeros((n, K)), np.zeros(n)
        for b in range(self.B):
            boot = rng.integers(n, size=n)
            t = DecisionTree(self.criterion, min_samples_leaf=self.min_samples_leaf, max_features=mf,
                             seed=int(rng.integers(2**31)), classes=self.classes_).fit(X[boot], y[boot])
            oob = np.setdiff1d(np.arange(n), boot)
            oob_sum[oob] += t.predict_proba(X[oob]); oob_cnt[oob] += 1
            self.trees.append(t); self.boots.append(boot)
        seen = oob_cnt > 0
        if self.regression:
            self.oob_score_ = np.mean((oob_sum[seen, 0] / oob_cnt[seen] - y[seen])**2)      # OOB MSE
        else:
            self.oob_score_ = np.mean(self.classes_[np.argmax(oob_sum[seen], axis=1)] != y[seen])   # OOB error
        return self

    def predict_proba(self, X, n_trees=None):
        return np.mean([t.predict_proba(X) for t in self.trees[:n_trees]], axis=0)

    def predict(self, X, n_trees=None):
        v = self.predict_proba(X, n_trees)
        return v[:, 0] if self.regression else self.classes_[np.argmax(v, axis=1)]

# ---------------------------------------------------------------- Sections 9-13: boosting with stumps
class StumpPool:
    # The finite set H of decision stumps phi(x) = +1 if x_j > theta else -1 (one per midpoint of each feature).
    # Negated stumps -phi are handled by the sign s chosen by the boosting algorithms.
    def __init__(self, X):
        feats, thrs = [], []
        for j in range(X.shape[1]):
            u = np.unique(X[:, j])
            mid = (u[1:] + u[:-1]) / 2
            feats.append(np.full(len(mid), j)); thrs.append(mid)
        self.feat, self.thr = np.concatenate(feats), np.concatenate(thrs)

    def phi(self, X, picks=None):
        f, t = (self.feat, self.thr) if picks is None else (self.feat[picks], self.thr[picks])
        return np.where(X[:, f] > t, 1.0, -1.0).astype(np.float32)

def adaboost(Phi, y, T):
    # AdaBoost (Mohri et al., Fig. 7.1) over the finite set {+-Phi[:, j]}, y in {-1, +1}.
    m = len(y)
    D = np.full(m, 1 / m)
    out = {k: [] for k in ['k', 's', 'alpha', 'eps', 'Z']}
    for t in range(T):
        r = (D * y) @ Phi                                   # r_j = 1 - 2 eps_j: the edge of each stump
        k = int(np.argmax(np.abs(r)))
        s = 1.0 if r[k] >= 0 else -1.0
        eps = (1 - abs(r[k])) / 2
        if eps < 1e-12:                                     # a perfect base classifier: alpha is infinite
            break
        alpha = 0.5 * np.log((1 - eps) / eps)
        Z = 2 * np.sqrt(eps * (1 - eps))
        D = D * np.exp(-alpha * y * s * Phi[:, k]) / Z
        D = D / D.sum()                                     # only removes floating-point drift
        for key, v in zip(out, [k, s, alpha, eps, Z]):
            out[key].append(v)
    return {key: np.array(v) for key, v in out.items()}

def staged_scores(model, Phi_selected):
    # f_t(x) for t = 1..T; Phi_selected = stumps of the rounds, one column per round
    return np.cumsum(model['alpha'] * model['s'] * Phi_selected, axis=1)

def coordinate_descent(H, y, T):
    # Maximum-descent coordinate descent on F, with a NUMERICAL line search (no AdaBoost formula inside).
    m, N = H.shape
    g, picks, steps, values = np.zeros(m), [], [], []
    for t in range(T):
        deriv = -(y * np.exp(-y * g)) @ H / m              # dF/d(alpha_k) for every k
        k = int(np.argmin(deriv))                           # steepest descent direction
        res = minimize_scalar(lambda eta: np.mean(np.exp(-y * (g + eta * H[:, k]))),
                              bounds=(0, 30), method='bounded', options={'xatol': 1e-10})
        g = g + res.x * H[:, k]
        picks.append(k); steps.append(res.x); values.append(np.mean(np.exp(-y * g)))
    return np.array(picks), np.array(steps), np.array(values)

def adaboost_mh(Phi, Y, T):
    # AdaBoost.MH (Mohri et al., Fig. 9.1) with base classifiers h(x, l) = c_l * phi(x):
    # one stump on x and one sign per label. D is a distribution over {1..m} x {1..k}; Y in {-1, +1}^(m x k).
    m, k = Y.shape
    D = np.full((m, k), 1 / (m * k))
    out = {key: [] for key in ['j', 'c', 'alpha', 'eps', 'Z']}
    for t in range(T):
        gamma = (D * Y).T @ Phi                             # (k, N): the edge per label of each stump
        r = np.abs(gamma).sum(axis=0)
        j = int(np.argmax(r))
        c = np.where(gamma[:, j] >= 0, 1.0, -1.0)           # the best sign for each label
        eps = (1 - r[j]) / 2
        if eps < 1e-12:
            break
        alpha = 0.5 * np.log((1 - eps) / eps)
        Z = 2 * np.sqrt(eps * (1 - eps))
        D = D * np.exp(-alpha * Y * (c[None, :] * Phi[:, j, None])) / Z
        D = D / D.sum()
        for key, v in zip(out, [j, c, alpha, eps, Z]):
            out[key].append(v)
    return {key: np.array(v) for key, v in out.items()}

def mh_scores(model, Phi_selected):
    # f(x, l) = sum_t alpha_t c_{t,l} phi_t(x); returns (n, k)
    return (Phi_selected * model['alpha']) @ model['c']

def labels_to_plus_minus(y, classes):
    # y_i[l] = +1 if l is the class of point i, -1 otherwise
    return np.where(y[:, None] == classes, 1.0, -1.0)

# ---------------------------------------------------------------- Section 14: gradient boosting
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

class GradientBoosting:
    # Gradient tree boosting (Hastie et al., Alg. 10.3) with shrinkage; loss: 'squared' or 'logistic' (y in {0, 1}).
    def __init__(self, loss='squared', n_estimators=100, lr=0.1, max_depth=3):
        self.loss, self.n_estimators, self.lr, self.max_depth = loss, n_estimators, lr, max_depth

    def fit(self, X, y):
        y = np.asarray(y, float)
        self.f0 = y.mean() if self.loss == 'squared' else np.log(y.mean() / (1 - y.mean()))
        f = np.full(len(y), self.f0)
        self.trees, self.leaf_values = [], []
        for _ in range(self.n_estimators):
            p = f if self.loss == 'squared' else sigmoid(f)
            r = y - p                                       # the negative gradient for both losses
            t = DecisionTree('mse', max_depth=self.max_depth).fit(X, r)
            leaves = t.apply(X)
            gamma = t.value[:, 0].copy()                    # squared loss: the leaf mean is optimal
            if self.loss == 'logistic':                     # one Newton step per leaf
                for l in np.unique(leaves):
                    in_l = leaves == l
                    gamma[l] = r[in_l].sum() / max((p[in_l] * (1 - p[in_l])).sum(), 1e-12)
            f = f + self.lr * gamma[leaves]
            self.trees.append(t); self.leaf_values.append(gamma)
        return self

    def decision_function(self, X, n_trees=None):
        f = np.full(len(X), self.f0)
        for t, g in list(zip(self.trees, self.leaf_values))[:n_trees]:
            f = f + self.lr * g[t.apply(X)]
        return f

    def predict(self, X, n_trees=None):
        f = self.decision_function(X, n_trees)
        return f if self.loss == 'squared' else (f > 0).astype(int)

# ---------------------------------------------------------------- Section 15: XGBoost from scratch
def xgb_grow(X, g, h, max_depth, lam, min_child_weight, lr):
    # Exact greedy XGBoost tree: returns a function x -> leaf weight (shrunk by lr)
    nodes = []
    def grow(idx, depth):
        G, H = g[idx].sum(), h[idx].sum()
        nid = len(nodes)
        nodes.append([-1, 0.0, -1, -1, -lr * G / (H + lam)])
        if depth >= max_depth:
            return nid
        n, best, best_j, best_thr = len(idx), 0.0, None, None
        for j in range(X.shape[1]):
            x = X[idx, j]
            order = np.argsort(x, kind='stable')
            xs = x[order]
            GL, HL = np.cumsum(g[idx][order])[:-1], np.cumsum(h[idx][order])[:-1]
            GR, HR = G - GL, H - HL
            valid = (xs[1:] > xs[:-1]) & (HL >= min_child_weight) & (HR >= min_child_weight)
            if not valid.any():
                continue
            gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G**2 / (H + lam))
            gain = np.where(valid, gain, -np.inf)
            k = np.argmax(gain)
            if gain[k] > best + 1e-12:
                best, best_j, best_thr = gain[k], j, xs[k] / 2 + xs[k + 1] / 2
        if best_j is None:
            return nid
        mask = X[idx, best_j] <= best_thr
        nodes[nid][:2] = [best_j, best_thr]
        nodes[nid][2] = grow(idx[mask], depth + 1)
        nodes[nid][3] = grow(idx[~mask], depth + 1)
        return nid
    grow(np.arange(len(X)), 0)
    return np.array(nodes)

def xgb_tree_predict(nodes, X):
    node, active = np.zeros(len(X), int), np.arange(len(X))
    while active.size:
        n = node[active]
        internal = nodes[n, 2] >= 0
        active, n = active[internal], n[internal]
        go_left = X[active, nodes[n, 0].astype(int)] <= nodes[n, 1]
        node[active] = np.where(go_left, nodes[n, 2], nodes[n, 3]).astype(int)
    return nodes[node, 4]

def softmax(F):
    F = F - F.max(axis=1, keepdims=True)
    P = np.exp(F)
    return P / P.sum(axis=1, keepdims=True)

class XGBScratch:
    # objective: 'binary' (logistic, y in {0, 1}) or 'softmax' (y in {0..K-1}, K trees per round).
    def __init__(self, objective='binary', n_rounds=20, lr=0.3, max_depth=3, lam=1.0, min_child_weight=1.0):
        self.objective, self.n_rounds, self.lr = objective, n_rounds, lr
        self.max_depth, self.lam, self.min_child_weight = max_depth, lam, min_child_weight

    def fit(self, X, y):
        grow = lambda g, h: xgb_grow(X, g, h, self.max_depth, self.lam, self.min_child_weight, self.lr)
        self.trees = []
        if self.objective == 'binary':
            f = np.zeros(len(y))
            for _ in range(self.n_rounds):
                p = sigmoid(f)
                t = grow(p - y, np.maximum(p * (1 - p), 1e-16))
                f = f + xgb_tree_predict(t, X)
                self.trees.append(t)
        else:
            self.K = int(y.max()) + 1
            Y, F = np.eye(self.K)[y], np.zeros((len(y), self.K))
            for _ in range(self.n_rounds):
                P = softmax(F)
                g, h = P - Y, np.maximum(2 * P * (1 - P), 1e-16)   # XGBoost's diagonal Hessian bound
                round_trees = [grow(g[:, c], h[:, c]) for c in range(self.K)]
                for c, t in enumerate(round_trees):
                    F[:, c] += xgb_tree_predict(t, X)
                self.trees.append(round_trees)
        return self

    def predict_proba(self, X):
        if self.objective == 'binary':
            return sigmoid(sum(xgb_tree_predict(t, X) for t in self.trees))
        F = np.zeros((len(X), self.K))
        for round_trees in self.trees:
            for c, t in enumerate(round_trees):
                F[:, c] += xgb_tree_predict(t, X)
        return softmax(F)

---
# Part I — Decision Trees

---
## 1. From Linear Models to Recursive Partitions

A **binary decision tree** (Mohri et al. [1], Def. 9.5) is a tree representation of a partition of the input space. Each interior node asks a question about the features — here the numerical question $x_j\le\theta$ — and each leaf carries a prediction. A point is routed from the root to one leaf; the leaves define **disjoint** regions $R_1,\dots,R_J$ that cover the space, and the model is

$$h(\mathbf{x})=\sum_{\ell=1}^J c_\ell\,\mathbb 1[\mathbf{x}\in R_\ell],$$

with $c_\ell$ the majority class of the training points in $R_\ell$ (classification) or their mean (regression, Section 4). All the regions are axis-aligned boxes.

Compared with the linear models of `06`–`08`:

* **Invariance.** Only the *order* of the values of each feature matters, so monotone rescaling changes nothing: no standardization is needed (contrast `06`, §15 and `07`, §14).
* **Interactions** are captured by the structure, without a feature map or a kernel.
* **Capacity grows with the tree.** A tree with enough leaves can realize any labeling of distinct points, so the VC dimension of unrestricted trees is infinite: the size of the tree has to be controlled (Section 5).

The cell below grows trees of increasing depth, using the `DecisionTree` class of the helper cell (derived in Sections 2–3).

In [ ]:
X, y = make_moons(150, 0.25, seed=0)
X_te, y_te = make_moons(2000, 0.25, seed=1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
print(f"{'depth':>6} {'leaves':>7} {'train acc':>10} {'test acc':>9}")
for ax, depth in zip(axes, [1, 2, 4, None]):
    t = DecisionTree('gini', max_depth=depth).fit(X, y)
    tr, te = np.mean(t.predict(X) == y), np.mean(t.predict(X_te) == y_te)
    plot_regions(ax, t.predict, X, y, f'max depth = {depth}\n{t.n_leaves()} leaves, train {tr:.3f}, test {te:.3f}')
    print(f"{str(depth):>6} {t.n_leaves():>7} {tr:>10.3f} {te:>9.3f}")
plt.suptitle('A Decision Tree Is a Partition Into Axis-Aligned Boxes', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2. Node Impurity: Misclassification, Entropy, Gini

To decide how to split, we need to measure how mixed a node is. For a node $n$ and class $l\in[k]$, let $p_l(n)$ be the fraction of its training points in class $l$. The three standard **impurity** measures [1, §9.3.3] are

$$
F(n)=\begin{cases}
1-\max_{l}p_l(n) & \text{misclassification}\\[2pt]
-\sum_{l}p_l(n)\log_2p_l(n) & \text{entropy}\\[2pt]
\sum_{l}p_l(n)\big(1-p_l(n)\big) & \text{Gini index}
\end{cases}
$$

A split $q$ sends a fraction $\eta$ of the points of $n$ to the left child $n_-$ and the rest to the right child $n_+$. The **impurity decrease** is

$$\tilde F(n,q)=F(n)-\big[\eta F(n_-)+(1-\eta)F(n_+)\big].$$

**The decrease is never negative.** All three measures are concave functions of the proportion vector $\mathbf{p}$, and $\mathbf{p}(n)=\eta\,\mathbf{p}(n_-)+(1-\eta)\,\mathbf{p}(n_+)$. Jensen's inequality then gives $F(n)\ge\eta F(n_-)+(1-\eta)F(n_+)$.

**Why not simply use misclassification?** It is only piecewise linear, so $\tilde F$ can be exactly zero although the split is clearly useful: this happens whenever the same class stays in the majority in both children. Entropy and Gini are *strictly* concave, so $\tilde F>0$ as soon as the children's proportions differ from the parent's (if they coincide, as in a split that carries no information, the decrease is zero for every measure). They are also differentiable, which is why they are preferred in practice. The two cells of the table below show the classic example: same misclassification decrease, different Gini and entropy decrease.

In [ ]:
p = np.linspace(0, 1, 501)
P = np.column_stack([p, 1 - p])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(p, impurity(P, 'misclassification'), color=BLUE, lw=2.5, label='misclassification')
ax.plot(p, 0.5 * impurity(P, 'entropy'), color=RED, lw=2.5, label='entropy (scaled by 0.5)')
ax.plot(p, impurity(P, 'gini'), color=TEAL, lw=2.5, label='Gini index')
ax.set_xlabel('fraction of class 1 in the node, $p$'); ax.set_ylabel('impurity')
ax.set_title('Impurity Measures, Two Classes', fontsize=12); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

def decrease(counts_left, counts_right, criterion):
    L, R = np.array(counts_left, float), np.array(counts_right, float)
    nl, nr = L.sum(), R.sum(); n = nl + nr
    return impurity((L + R) / n, criterion) - (nl / n * impurity(L / nl, criterion) + nr / n * impurity(R / nr, criterion))

# A node with 400 points of each class, and two candidate splits
splits = {'A: (300,100) | (100,300)': ((300, 100), (100, 300)),
          'B: (200,400) | (200,  0)': ((200, 400), (200, 0))}
print(f"{'split':28s} {'misclass.':>10} {'entropy':>9} {'Gini':>8}")
for name, (l, r) in splits.items():
    print(f"{name:28s} {decrease(l, r, 'misclassification'):>10.4f} {decrease(l, r, 'entropy'):>9.4f} {decrease(l, r, 'gini'):>8.4f}")

# Misclassification can see no gain at all: the majority class is the same in both children
l, r = (40, 10), (45, 5)
print(f"\nchildren {l} | {r}: misclassification decrease = {decrease(l, r, 'misclassification'):.4f}, Gini decrease = {decrease(l, r, 'gini'):.4f}")

# Concavity: the decrease is never negative, on 100,000 random splits of random nodes
rng = np.random.default_rng(0)
L, R = rng.integers(1, 200, (100_000, 3)).astype(float), rng.integers(1, 200, (100_000, 3)).astype(float)
for crit in ['misclassification', 'entropy', 'gini']:
    nl, nr = L.sum(1), R.sum(1); n = nl + nr
    d = impurity((L + R) / n[:, None], crit) - (nl / n * impurity(L / nl[:, None], crit) + nr / n * impurity(R / nr[:, None], crit))
    print(f"min decrease over 100,000 random splits, {crit:18s}: {d.min():+.2e}")